<a href="https://colab.research.google.com/github/ionikim/epilepsy_pediatrics_EEG/blob/main/src/01_loading/preprocessing_180326_JO.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In this file, we
1. automate the data preprocessing and
2. create multiplex horizontal visibility graphs for a certain time window in a file (edge list + adjacency matrix)

Comments:
- For the code to work, it needs to be in the same folder as the .edf and .seizure files.
- Window size can be adjusted; in my case the computer didn't have enough RAM for more than 5s.

What is missing:
- the code does not process files without a corresponding .seizure file
- actual graph creation for the correct windows (one wictal and one interictal)

In [32]:
!pip install ts2vg
!pip install mne
%cd /content
!git clone https://github.com/ionikim/epilepsy_pediatrics_EEG.git
%cd /content/epilepsy_pediatrics_EEG
from pathlib import Path
import mne
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import os
import glob
from ts2vg import HorizontalVG
import scipy.sparse as sp
import json

/content
fatal: destination path 'epilepsy_pediatrics_EEG' already exists and is not an empty directory.
/content/epilepsy_pediatrics_EEG


In [2]:
RAW_DIR = Path("/data/raw")

PROCESSED_DIR = Path("/data/processed")
LABELED_DIR   = PROCESSED_DIR / "labeled_signals"
FILEMETA_DIR  = PROCESSED_DIR / "file_metadata"

WINDOWS_DIR      = Path("/data/windows")
WINDOWSIG_DIR    = WINDOWS_DIR / "signals"
WINDOWMETA_DIR   = WINDOWS_DIR / "metadata"

for d in [LABELED_DIR, FILEMETA_DIR, WINDOWSIG_DIR, WINDOWMETA_DIR]:
    d.mkdir(parents=True, exist_ok=True)

In [33]:
print("CWD:", Path.cwd())
print("RAW_DIR exists:", Path("data/raw").exists())
print("Files in raw:")
for p in Path("data/raw").glob("*"):
    print(p.name)

CWD: /content/epilepsy_pediatrics_EEG
RAW_DIR exists: True
Files in raw:
.gitkeep
chb01_03.edf
chb01_03.edf.seizures


In [34]:
def save_json(data, out_path):
    with open(out_path, "w", encoding="utf-8") as f:
        json.dump(data, f, indent=2, ensure_ascii=False)

In [48]:
PROCESSED_DIR = Path("data/processed")

# Extract seizure start and length from a .seizures annotation file.
def get_seizure_period(annotation_file):
    with open(annotation_file, "rb") as f:
        byte_array = f.read()
    start = int(bin(byte_array[38])[2:] + bin(byte_array[41])[2:], 2)
    length = byte_array[49]
    return start, length


# Load an EDF file, create a DataFrame, and label seizure periods
def process_edf_with_labels(edf_file, seizure_file):
    raw = mne.io.read_raw_edf(str(edf_file), preload=True, verbose=False)
    raw.filter(1., 45.)
    raw.set_eeg_reference('average')

    electrode_names = raw.ch_names
    sfreq = raw.info['sfreq']
    n_samples = len(raw.times)
    time_marks = np.arange(n_samples) / sfreq
    electrode_data = raw.get_data()

    df = pd.DataFrame(electrode_data.T, columns=electrode_names)
    df.index = time_marks
    df.index.name = 'Time (s)'

    seizure_start = None
    seizure_length = None
    seizure_end = None

    if seizure_file:
        seizure_start, seizure_length = get_seizure_period(seizure_file)
        seizure_end = seizure_start + seizure_length
        df["label"] = df.index.map(
            lambda t: "ictal" if seizure_start <= t <= seizure_end else "interictal"
        )
        has_seizure = True
    else:
        df["label"] = "interictal"
        has_seizure = False

    file_meta = {
        "edf_file": str(edf_file),
        "seizure_file": str(seizure_file) if seizure_file else None,
        "sampling_rate_hz": float(sfreq),
        "channel_names": list(electrode_names),
        "n_channels": int(len(electrode_names)),
        "n_samples": int(n_samples),
        "duration_s": float(n_samples / sfreq),
        "has_seizure": bool(has_seizure),
        "seizure_start_s": float(seizure_start) if seizure_start is not None else None,
        "seizure_length_s": float(seizure_length) if seizure_length is not None else None,
        "seizure_end_s": float(seizure_end) if seizure_end is not None else None,
    }

    return df, file_meta

In [52]:
def extract_window(df, window_size_s=10, mode="ictal"):
    sfreq = int(round(1 / (df.index[1] - df.index[0])))
    window_size_samples = window_size_s * sfreq

    if mode == "ictal":
        indices = np.where(df["label"] == "ictal")[0]
    elif mode == "interictal":
        indices = np.where(df["label"] == "interictal")[0]
    else:
        raise ValueError("mode must be 'ictal' or 'interictal'")

    valid_starts = [
        idx for idx in indices
        if idx + window_size_samples <= len(df)
        and (df["label"].iloc[idx : idx + window_size_samples] == mode).all()
    ]

    if not valid_starts:
        raise ValueError(f"No contiguous {mode} block of {window_size_s}s found.")

    # deterministic selection
    if mode == "ictal":
        # onset-based: earliest valid ictal start
        start_idx = int(valid_starts[0])
    elif mode == "interictal":
        # fixed interictal: earliest valid interictal start
        start_idx = int(valid_starts[0])

    window = df.iloc[start_idx : start_idx + window_size_samples]
    return window, start_idx

def process_all_files(pattern="*.seizures", window_size_s=10):
    seizure_files = sorted(glob.glob(pattern))

    if not seizure_files:
        print(f"No files found matching: {pattern}")
        return {}, {}, pd.DataFrame(), pd.DataFrame()

    all_dataframes = {}
    windows = {}
    window_records = []
    log_records = []

    for i, seizure_file in enumerate(seizure_files):
        edf_file = seizure_file.replace(".seizures", "")

        try:
            # 1. full dataframe
            df, file_meta = process_edf_with_labels(edf_file, seizure_file)
            all_dataframes[edf_file] = df

            # full labeled df 저장
            file_stem = Path(edf_file).stem
            out_df = Path("data/processed/labeled_signals") / f"{file_stem}.parquet"
            out_df.parent.mkdir(parents=True, exist_ok=True)
            df.to_parquet(out_df)

            # 2. ictal / interictal window
            ictal_window, ictal_start_idx = extract_window(df, window_size_s=window_size_s, mode="ictal")
            interictal_window, interictal_start_idx = extract_window(df, window_size_s=window_size_s, mode="interictal")

            # 3. windows dict
            ictal_name = f"window_ictal_{i}"
            interictal_name = f"window_interictal_{i}"

            windows[ictal_name] = ictal_window
            windows[interictal_name] = interictal_window

            # window 저장
            win_dir = Path("data/windows/signals")
            win_dir.mkdir(parents=True, exist_ok=True)
            ictal_out = win_dir / f"{ictal_name}.parquet"
            interictal_out = win_dir / f"{interictal_name}.parquet"

            ictal_window.to_parquet(ictal_out)
            interictal_window.to_parquet(interictal_out)

            # 4. metadata rows
            sfreq = int(round(1 / (df.index[1] - df.index[0])))

            window_records.append({
              "window_id": ictal_name,
              "source_edf": edf_file,
              "label": "ictal",
              "selection_rule": "onset",
              "start_idx": ictal_start_idx,
              "end_idx": ictal_start_idx + len(ictal_window) - 1,
              "start_time_s": float(ictal_window.index[0]),
              "end_time_s": float(ictal_window.index[-1]),
              "n_samples": len(ictal_window),
              "window_size_s": window_size_s,
              "sampling_rate_hz": sfreq
})
            window_records.append({
              "window_id": interictal_name,
              "source_edf": edf_file,
              "label": "interictal",
              "selection_rule": "first_valid_interictal",
              "start_idx": interictal_start_idx,
              "end_idx": interictal_start_idx + len(interictal_window) - 1,
              "start_time_s": float(interictal_window.index[0]),
              "end_time_s": float(interictal_window.index[-1]),
              "n_samples": len(interictal_window),
              "window_size_s": window_size_s,
              "sampling_rate_hz": sfreq
})

            # 5. success log
            log_records.append({
                "edf_file": edf_file,
                "status": "success",
                "message": "ok",
                "saved_labeled_df": str(out_df),
                "saved_ictal_window": str(ictal_out),
                "saved_interictal_window": str(interictal_out)
            })

            # 6. summary
            print(f"\n[{i}] {edf_file}")
            print(f"  Has seizure              : {file_meta['has_seizure']}")
            print(f"  Saved full df            : {out_df}")

            print(f"  Ictal window name        : {ictal_name}")
            print(f"  Ictal start idx          : {ictal_start_idx}")
            print(f"  Ictal end idx            : {ictal_start_idx + len(ictal_window) - 1}")
            print(f"  Ictal start time (s)     : {float(ictal_window.index[0])}")
            print(f"  Ictal end time (s)       : {float(ictal_window.index[-1])}")

            print(f"  Interictal window name   : {interictal_name}")
            print(f"  Interictal start idx     : {interictal_start_idx}")
            print(f"  Interictal end idx       : {interictal_start_idx + len(interictal_window) - 1}")
            print(f"  Interictal start time(s) : {float(interictal_window.index[0])}")
            print(f"  Interictal end time (s)  : {float(interictal_window.index[-1])}")

        except FileNotFoundError as e:
            print(f"ERROR: EDF file not found for {seizure_file} — expected {edf_file}")
            log_records.append({
                "edf_file": edf_file,
                "status": "file_not_found",
                "message": str(e)
            })

        except ValueError as e:
            print(f"WARNING [{edf_file}]: {e}")
            log_records.append({
                "edf_file": edf_file,
                "status": "warning",
                "message": str(e)
            })

        except Exception as e:
            print(f"ERROR processing {seizure_file}: {e}")
            log_records.append({
                "edf_file": edf_file,
                "status": "error",
                "message": str(e)
            })

    windows_index = pd.DataFrame(window_records)
    out_index = Path("data/windows/metadata/windows_index.csv")
    out_index.parent.mkdir(parents=True, exist_ok=True)
    windows_index.to_csv(out_index, index=False)

    processing_log = pd.DataFrame(log_records)
    out_log = PROCESSED_DIR / "processing_log.csv"
    out_log.parent.mkdir(parents=True, exist_ok=True)
    processing_log.to_csv(out_log, index=False)

    print(f"\nSaved window index: {out_index}")
    print(f"Saved processing log: {out_log}")

    return all_dataframes, windows, windows_index, processing_log

In [53]:
# Run it
all_dataframes, windows, windows_index, processing_log = process_all_files(
    pattern="data/raw/*.edf.seizures",
    window_size_s=5
)

# Access individual windows
# windows["window_ictal_0"]
# windows["window_interictal_2"]

# Check if all windows have the same number of samples
sizes = {name: len(df) for name, df in windows.items()}
print("\nWindow sizes:", sizes)

/tmp/ipykernel_45652/435486982.py:14: RuntimeWarning: Channel names are not unique, found duplicates for: {'T8-P8'}. Applying running numbers for duplicates.
  raw = mne.io.read_raw_edf(str(edf_file), preload=True, verbose=False)


Filtering raw data in 1 contiguous segment
Setting up band-pass filter from 1 - 45 Hz

FIR filter parameters
---------------------
Designing a one-pass, zero-phase, non-causal bandpass filter:
- Windowed time-domain design (firwin) method
- Hamming window with 0.0194 passband ripple and 53 dB stopband attenuation
- Lower passband edge: 1.00
- Lower transition bandwidth: 1.00 Hz (-6 dB cutoff frequency: 0.50 Hz)
- Upper passband edge: 45.00 Hz
- Upper transition bandwidth: 11.25 Hz (-6 dB cutoff frequency: 50.62 Hz)
- Filter length: 845 samples (3.301 s)

EEG channel type selected for re-referencing
Applying average reference.
Applying a custom ('EEG',) reference.

[0] data/raw/chb01_03.edf
  Has seizure              : True
  Saved full df            : data/processed/labeled_signals/chb01_03.parquet
  Ictal window name        : window_ictal_0
  Ictal start idx          : 766976
  Ictal end idx            : 768255
  Ictal start time (s)     : 2996.0
  Ictal end time (s)       : 3000.9960

In [24]:
#display(windows["window_ictal_0"])
#display(windows["window_interictal_0"])

In [61]:
# build a multiplex horizontal visibility graph from an EEG window df
# -> each electrode is on one layer, each time point connects all electrodes (inter-layer edges)
# returns: edge list, adjacency matrix, node_index (dict for global node index), layer_info (dict with per-electrode HVG info)

def build_multiplex_hvg(window_df):
    electrode_cols = [c for c in window_df.columns if c != "label"]
    n_electrodes   = len(electrode_cols)
    n_timepoints   = len(window_df)
    n_nodes_total  = n_electrodes * n_timepoints

    print(f"Electrodes   : {n_electrodes}")
    print(f"Time points  : {n_timepoints}")
    print(f"Total nodes  : {n_nodes_total}")

    # 1. NODE INDEX: map (electrode_idx, time_idx) -> global node index
    # node layout: electrode 0 gets nodes 0..n_timepoints-1,
    #              electrode 1 gets nodes n_timepoints..2*n_timepoints-1, etc.

    def node_id(layer, t):
        return layer * n_timepoints + t

    node_index = {
        (electrode_cols[l], t): node_id(l, t)
        for l in range(n_electrodes)
        for t in range(n_timepoints)
    }

    # 2. INTRA-LAYER EDGES: HVG edges within each electrode
    intra_edges = []
    layer_info  = {}

    for l, electrode in enumerate(electrode_cols):
        ts  = window_df[electrode].values
        hvg = HorizontalVG()
        hvg.build(ts)

        for (t_i, t_j) in hvg.edges:
            u = node_id(l, t_i)
            v = node_id(l, t_j)
            intra_edges.append({
                "source"       : u,
                "target"       : v,
                "source_label" : f"{electrode}_t{t_i}",
                "target_label" : f"{electrode}_t{t_j}",
                "edge_type"    : "intra",
                "layer"        : electrode,
            })

        layer_info[electrode] = {
            "n_intra_edges": len(hvg.edges),
            "layer_index"  : l,
        }

    print(f"Intra-layer edges : {len(intra_edges)}")

    # 3. INTER-LAYER EDGES: connect same time point across all electrodes
    # at each time point t, connect every electrode to every other (full coupling)

    inter_edges = []

    for t in range(n_timepoints):
        for l_i in range(n_electrodes):
            for l_j in range(l_i + 1, n_electrodes):   # upper triangle only (undirected)
                u = node_id(l_i, t)
                v = node_id(l_j, t)
                inter_edges.append({
                    "source"       : u,
                    "target"       : v,
                    "source_label" : f"{electrode_cols[l_i]}_t{t}",
                    "target_label" : f"{electrode_cols[l_j]}_t{t}",
                    "edge_type"    : "inter",
                    "layer"        : f"{electrode_cols[l_i]}<->{electrode_cols[l_j]}",
                })

    print(f"Inter-layer edges : {len(inter_edges)}")

    # 4. COMBINED EDGE LIST
    edge_list = pd.DataFrame(intra_edges + inter_edges)
    print(f"Total edges       : {len(edge_list)}")

    # 5. ADJACENCY MATRIX
    # use sparse matrix for efficiency (n_nodes can be large)
    adj_sparse = sp.lil_matrix((n_nodes_total, n_nodes_total), dtype=np.int8)

    for _, row in edge_list.iterrows():
        u, v = int(row["source"]), int(row["target"])
        adj_sparse[u, v] = 1
        adj_sparse[v, u] = 1  # undirected

    # convert to labeled DataFrame
    node_labels = [
        f"{electrode_cols[l]}_t{t}"
        for l in range(n_electrodes)
        for t in range(n_timepoints)
    ]
    adj_matrix = pd.DataFrame(
        adj_sparse.toarray(),
        index   = node_labels,
        columns = node_labels
    )

    return edge_list, adj_matrix, node_index, layer_info


# Run on first ictal window
ictal_key = next(k for k in windows if k.startswith("window_ictal_"))
window    = windows[ictal_key]

edge_list, adj_matrix, node_index, layer_info = build_multiplex_hvg(window)

graph_id = f"chb01_03_{ictal_key}"

saved_paths = save_multiplex_hvg_outputs(
    edge_list=edge_list,
    adj_matrix=adj_matrix,
    node_index=node_index,
    layer_info=layer_info,
    window_df=window,
    graph_id=graph_id,
    source_edf="data/raw/chb01_03.edf",
    window_id=ictal_key,
    label="ictal",
    out_root="data/graphs"
)

interictal_key = next(k for k in windows if k.startswith("window_interictal_"))
window = windows[interictal_key]

edge_list, adj_matrix, node_index, layer_info = build_multiplex_hvg(window)

graph_id = f"chb01_03_{interictal_key}"

saved_paths_interictal = save_multiplex_hvg_outputs(
    edge_list=edge_list,
    adj_matrix=adj_matrix,
    node_index=node_index,
    layer_info=layer_info,
    window_df=window,
    graph_id=graph_id,
    source_edf="data/raw/chb01_03.edf",
    window_id=interictal_key,
    label="interictal",
    out_root="data/graphs"
)

# inspect results
print("\n--- Edge list (first 10) ---")
display(edge_list.head(10))

print("\n--- Adjacency matrix (first 5x5) ---")
display(adj_matrix.iloc[:5, :5])

print("\n--- Layer info ---")
display(pd.DataFrame(layer_info).T)

# split edge list back into intra / inter if needed
intra = edge_list[edge_list["edge_type"] == "intra"]
inter = edge_list[edge_list["edge_type"] == "inter"]

Electrodes   : 23
Time points  : 1280
Total nodes  : 29440
Intra-layer edges : 57801
Inter-layer edges : 323840
Total edges       : 381641
Saved edge list : data/graphs/edge_lists/chb01_03_window_ictal_0_edge_list.parquet
Saved node table: data/graphs/node_tables/chb01_03_window_ictal_0_nodes.parquet
Saved layer info: data/graphs/layer_info/chb01_03_window_ictal_0_layer_info.csv
Saved adjacency : data/graphs/adjacency/chb01_03_window_ictal_0_adj_sparse.npz
Saved metadata  : data/graphs/metadata/chb01_03_window_ictal_0_graph_meta.json
Electrodes   : 23
Time points  : 1280
Total nodes  : 29440
Intra-layer edges : 57827
Inter-layer edges : 323840
Total edges       : 381667
Saved edge list : data/graphs/edge_lists/chb01_03_window_interictal_0_edge_list.parquet
Saved node table: data/graphs/node_tables/chb01_03_window_interictal_0_nodes.parquet
Saved layer info: data/graphs/layer_info/chb01_03_window_interictal_0_layer_info.csv
Saved adjacency : data/graphs/adjacency/chb01_03_window_interic

,source,target,source_label,target_label,edge_type,layer
0,53,54,FP1-F7_t53,FP1-F7_t54,intra,FP1-F7
1,54,55,FP1-F7_t54,FP1-F7_t55,intra,FP1-F7
2,52,53,FP1-F7_t52,FP1-F7_t53,intra,FP1-F7
3,55,56,FP1-F7_t55,FP1-F7_t56,intra,FP1-F7
4,51,52,FP1-F7_t51,FP1-F7_t52,intra,FP1-F7
5,48,52,FP1-F7_t48,FP1-F7_t52,intra,FP1-F7
6,56,57,FP1-F7_t56,FP1-F7_t57,intra,FP1-F7
7,56,257,FP1-F7_t56,FP1-F7_t257,intra,FP1-F7
8,56,258,FP1-F7_t56,FP1-F7_t258,intra,FP1-F7
9,56,705,FP1-F7_t56,FP1-F7_t705,intra,FP1-F7



--- Adjacency matrix (first 5x5) ---


,FP1-F7_t0,FP1-F7_t1,FP1-F7_t2,FP1-F7_t3,FP1-F7_t4
FP1-F7_t0,0,1,0,0,0
FP1-F7_t1,1,0,1,0,0
FP1-F7_t2,0,1,0,1,0
FP1-F7_t3,0,0,1,0,1
FP1-F7_t4,0,0,0,1,0



--- Layer info ---


,n_intra_edges,layer_index
FP1-F7,2517,0
F7-T7,2485,1
T7-P7,2513,2
P7-O1,2514,3
FP1-F3,2509,4
F3-C3,2482,5
C3-P3,2515,6
P3-O1,2510,7
FP2-F4,2517,8
F4-C4,2539,9
